# OmniCare AI: referência de marcha normal (Multi-Gait-Posture)

Este notebook aprende como é a marcha de pessoas saudáveis usando um andador e, a partir disso, detecta caminhadas fora do padrão. Ele atende ao requisito de anomalias em **padrões de movimentação do paciente**.

## Como executar

1. Confira se `multi-gait-posture-1.0.0.zip` está em `Meu Drive/omnicare/`.
2. Abra este notebook com o **Google Colaboratory**. Não precisa de GPU.
3. Use **Ambiente de execução > Executar tudo** e autorize o acesso ao Drive.

O notebook lê do zip só os 14 arquivos de esqueleto (cerca de 212 MB dos 19,4 GB), sem extrair o restante.

## O que os resultados significam

- A base tem só pessoas saudáveis, então **não existem casos reais de marcha alterada** nela.
- As anomalias usadas no teste são **simuladas**, e cada uma altera justamente o que as medidas observam. Por isso, a detecção mostrada aqui é otimista e não prova desempenho em pacientes.
- O teste sempre usa uma pessoa que ficou fora do treino.

Licença da base: Creative Commons Attribution 4.0. Cite conforme o README do projeto.

## 1. Ambiente

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Configuração

In [ ]:
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/omnicare')
ZIP_PATH = DRIVE_DIR / 'multi-gait-posture-1.0.0.zip'
OUTPUT_DIR = DRIVE_DIR / 'resultados_marcha'

WINDOW_SECONDS = 8
HOP_SECONDS = 4
FALSE_ALARM_TARGET = 0.05  # fração de caminhadas normais que pode passar do limiar
SEED = 42

In [ ]:
import io
import json
import zipfile

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks
from sklearn.covariance import LedoitWolf
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FS = 30  # o esqueleto processado tem 30 amostras por segundo
JOINTS = (
    'pelvis', 'l5', 'l3', 't12', 't8', 'neck', 'head', 'right_shoulder', 'right_upper_arm', 'right_forearm',
    'right_hand', 'left_shoulder', 'left_upper_arm', 'left_forearm', 'left_hand', 'right_upper_leg',
    'right_lower_leg', 'right_foot', 'right_toe', 'left_upper_leg', 'left_lower_leg', 'left_foot', 'left_toe',
)
JOINT = {name: index for index, name in enumerate(JOINTS)}
UPPER_BODY = [JOINT[name] for name in JOINTS[1:15]]  # tudo acima da pelve
WINDOW = WINDOW_SECONDS * FS
HOP = HOP_SECONDS * FS
FILTER_B, FILTER_A = butter(4, 6 / (FS / 2))  # remove oscilações acima de 6 Hz

## 3. Dados

In [ ]:
if not ZIP_PATH.exists():
    raise FileNotFoundError(f'Não encontrei {ZIP_PATH}. Confira a pasta e o nome do arquivo no Drive.')


def load_trials(path):
    trials = []
    with zipfile.ZipFile(path) as outer:
        names = sorted(n for n in outer.namelist() if '/processed_data/participant' in n and n.endswith('.zip'))
        for name in names:
            with zipfile.ZipFile(io.BytesIO(outer.read(name))) as inner:
                for item in sorted(inner.namelist()):
                    if item.split('/')[-1] != 'normalized_skeleton_3d.csv':
                        continue
                    _, sequence, speed, place = item.split('/')[-2].split('_')
                    values = pd.read_csv(io.BytesIO(inner.read(item)), index_col=0).to_numpy(dtype=np.float64)
                    trials.append({
                        'id': len(trials),
                        'pid': item.split('/')[0],
                        'sequence': sequence,
                        'speed': float(speed),
                        'place': place,
                        'X': values.reshape(-1, len(JOINTS), 3),  # eixos: x frente, y esquerda, z cima
                    })
            print('Lido:', name.split('/')[-1], '| caminhadas até agora:', len(trials))
    return trials


trials = load_trials(ZIP_PATH)
participants = sorted({t['pid'] for t in trials})
assert len(participants) == 14, f'Esperava 14 participantes e encontrei {len(participants)}.'
print('Participantes:', len(participants), '| caminhadas:', len(trials))

## 4. Medidas de marcha

Cada caminhada é dividida em janelas de 8 segundos. Os passos são localizados pela distância para frente e para trás entre os calcanhares. Comprimentos são divididos pelo tamanho da perna, para comparar pessoas de alturas diferentes.

In [ ]:
MEASURES = {
    'passo_medio': 'Comprimento médio do passo, em comprimentos de perna',
    'assimetria_passo': 'Diferença entre os passos esquerdo e direito',
    'assimetria_tempo': 'Diferença entre os tempos dos passos esquerdo e direito',
    'cadencia': 'Passos por minuto',
    'cv_tempo': 'Irregularidade do tempo dos passos',
    'cv_passo': 'Irregularidade do comprimento dos passos',
    'largura_passo': 'Distância lateral entre os calcanhares, em comprimentos de perna',
    'tronco_frente_media': 'Inclinação média do tronco para frente, em graus',
    'tronco_frente_dp': 'Oscilação do tronco para frente e para trás, em graus',
    'tronco_lado_abs': 'Inclinação média do tronco para o lado, em graus',
    'tronco_lado_dp': 'Oscilação lateral do tronco, em graus',
    'joelho_amplitude': 'Amplitude de movimento do joelho, em graus',
    'assimetria_joelho': 'Diferença de amplitude entre os joelhos',
}
NAMES = list(MEASURES)


def joint_angle(a, b, c):
    u, v = a - b, c - b
    cosine = (u * v).sum(axis=1) / (np.linalg.norm(u, axis=1) * np.linalg.norm(v, axis=1))
    return np.degrees(np.arccos(np.clip(cosine, -1, 1)))


def window_measures(X):
    X = filtfilt(FILTER_B, FILTER_A, X, axis=0)
    hip_l, knee_l, heel_l = (X[:, JOINT[f'left_{n}']] for n in ('upper_leg', 'lower_leg', 'foot'))
    hip_r, knee_r, heel_r = (X[:, JOINT[f'right_{n}']] for n in ('upper_leg', 'lower_leg', 'foot'))
    leg = np.mean([
        np.linalg.norm(hip_l - knee_l, axis=1).mean() + np.linalg.norm(knee_l - heel_l, axis=1).mean(),
        np.linalg.norm(hip_r - knee_r, axis=1).mean() + np.linalg.norm(knee_r - heel_r, axis=1).mean(),
    ])

    separation = (heel_l[:, 0] - heel_r[:, 0]) / leg
    left, _ = find_peaks(separation, distance=int(0.4 * FS), prominence=0.15)
    right, _ = find_peaks(-separation, distance=int(0.4 * FS), prominence=0.15)
    if len(left) < 2 or len(right) < 2:
        return None  # poucos passos na janela

    events = sorted([(i, 'L') for i in left] + [(i, 'R') for i in right])
    step_times, left_times, right_times = [], [], []
    for (start, start_side), (end, end_side) in zip(events, events[1:]):
        step_times.append((end - start) / FS)
        if start_side != end_side:
            (left_times if end_side == 'L' else right_times).append((end - start) / FS)
    if not left_times or not right_times:
        return None

    left_lengths, right_lengths = separation[left], -separation[right]
    all_lengths = np.concatenate([left_lengths, right_lengths])
    mean_length = (left_lengths.mean() + right_lengths.mean()) / 2
    left_time, right_time = np.mean(left_times), np.mean(right_times)

    spine = X[:, JOINT['neck']] - X[:, JOINT['pelvis']]
    trunk_front = np.degrees(np.arctan2(spine[:, 0], spine[:, 2]))
    trunk_side = np.degrees(np.arctan2(spine[:, 1], spine[:, 2]))
    knee_left = np.ptp(joint_angle(hip_l, knee_l, heel_l))
    knee_right = np.ptp(joint_angle(hip_r, knee_r, heel_r))

    return {
        'passo_medio': mean_length,
        'assimetria_passo': abs(left_lengths.mean() - right_lengths.mean()) / mean_length,
        'assimetria_tempo': abs(left_time - right_time) / ((left_time + right_time) / 2),
        'cadencia': 60 / np.mean(step_times),
        'cv_tempo': np.std(step_times) / np.mean(step_times),
        'cv_passo': all_lengths.std() / all_lengths.mean(),
        'largura_passo': np.abs(heel_l[:, 1] - heel_r[:, 1]).mean() / leg,
        'tronco_frente_media': trunk_front.mean(),
        'tronco_frente_dp': trunk_front.std(),
        'tronco_lado_abs': abs(trunk_side.mean()),
        'tronco_lado_dp': trunk_side.std(),
        'joelho_amplitude': (knee_left + knee_right) / 2,
        'assimetria_joelho': abs(knee_left - knee_right) / ((knee_left + knee_right) / 2),
    }


def walk_windows(X):
    rows = []
    for start in range(0, X.shape[0] - WINDOW + 1, HOP):
        measures = window_measures(X[start:start + WINDOW])
        if measures is not None:
            rows.append(measures)
    return rows


windows_df = pd.DataFrame([
    {'pid': t['pid'], 'trial': t['id'], 'sequence': t['sequence'], 'speed': t['speed'], **measures}
    for t in trials
    for measures in walk_windows(t['X'])
])
straight_df = windows_df[windows_df['sequence'] == 'straight'].reset_index(drop=True)
print('Janelas:', windows_df.groupby('sequence').size().to_dict())
print('Caminhadas sem janela válida:', len(trials) - windows_df['trial'].nunique())
print('\nMedianas em linha reta, por velocidade (m/s):')
print(straight_df.groupby('speed')[NAMES].median().round(3).T.to_string())

## 5. Anomalias simuladas

Como a base só tem pessoas saudáveis, o teste altera as caminhadas de forma controlada. As alterações são testes de engenharia, não modelos clínicos de nenhuma doença.

In [ ]:
def rotate_upper_body(X, forward=None, sideways=None):
    out = X.copy()
    pelvis = X[:, [JOINT['pelvis']]]
    relative = X[:, UPPER_BODY] - pelvis
    if forward is not None:
        angle = np.radians(forward) * np.ones(len(X))
        cos, sin = np.cos(angle)[:, None], np.sin(angle)[:, None]
        x, z = relative[..., 0].copy(), relative[..., 2].copy()
        relative[..., 0], relative[..., 2] = cos * x + sin * z, -sin * x + cos * z
    if sideways is not None:
        angle = np.radians(sideways) * np.ones(len(X))
        cos, sin = np.cos(angle)[:, None], np.sin(angle)[:, None]
        y, z = relative[..., 1].copy(), relative[..., 2].copy()
        relative[..., 1], relative[..., 2] = cos * y - sin * z, sin * y + cos * z
    out[:, UPPER_BODY] = relative + pelvis
    return out


def shorten_left_step(X, factor):
    out = X.copy()
    hip = X[:, [JOINT['left_upper_leg']]]
    leg_joints = [JOINT['left_lower_leg'], JOINT['left_foot'], JOINT['left_toe']]
    out[:, leg_joints, 0] = hip[..., 0] + (X[:, leg_joints, 0] - hip[..., 0]) * factor
    return out


def stretch_one_step(X, factor):
    smooth = filtfilt(FILTER_B, FILTER_A, X, axis=0)
    separation = smooth[:, JOINT['left_foot'], 0] - smooth[:, JOINT['right_foot'], 0]
    left, _ = find_peaks(separation, distance=int(0.4 * FS), prominence=0.1)
    right, _ = find_peaks(-separation, distance=int(0.4 * FS), prominence=0.1)
    events = sorted([(i, 'L') for i in left] + [(i, 'R') for i in right])
    if len(events) < 3:
        return X
    source, target, now, previous = [0.0], [0.0], 0.0, 0
    for index, side in events + [(len(X) - 1, 'E')]:
        now += (index - previous) * (factor if side == 'R' else 1.0)
        source.append(float(index))
        target.append(now)
        previous = index
    new_times = np.interp(np.arange(int(target[-1]) + 1), target, source)
    flat = X.reshape(len(X), -1)
    original_times = np.arange(len(X))
    stretched = np.column_stack([np.interp(new_times, original_times, flat[:, k]) for k in range(flat.shape[1])])
    return stretched.reshape(-1, len(JOINTS), 3)


def lateral_sway(X, degrees):
    return rotate_upper_body(X, sideways=degrees * np.sin(2 * np.pi * 0.5 * np.arange(len(X)) / FS))


SIMULATIONS = {
    'tronco 10 graus a mais para frente': lambda X: rotate_upper_body(X, forward=10),
    'tronco 20 graus a mais para frente': lambda X: rotate_upper_body(X, forward=20),
    'balanço lateral de 5 graus': lambda X: lateral_sway(X, 5),
    'balanço lateral de 10 graus': lambda X: lateral_sway(X, 10),
    'perna esquerda com passo 20% menor': lambda X: shorten_left_step(X, 0.8),
    'perna esquerda com passo 40% menor': lambda X: shorten_left_step(X, 0.6),
    'um passo 30% mais lento': lambda X: stretch_one_step(X, 1.3),
    'um passo 60% mais lento': lambda X: stretch_one_step(X, 1.6),
}

straight_trials = [t for t in trials if t['sequence'] == 'straight']
simulated = {
    name: pd.DataFrame([
        {'pid': t['pid'], 'trial': t['id'], **measures}
        for t in straight_trials
        for measures in walk_windows(change(t['X']))
    ])
    for name, change in SIMULATIONS.items()
}
print({name: len(df) for name, df in simulated.items()})

## 6. Comparação dos métodos

O teste deixa uma pessoa de fora por vez, e o limiar é definido sem olhar essa pessoa. A pontuação de cada caminhada é a mediana das suas janelas. A referência é aprendida só em linha reta, e o resultado em curvas mede quantos alarmes falsos surgem em uma situação não vista no treino.

- **Pontuação z robusta:** mede o quanto cada medida se afasta do normal e usa a maior distância. Mostra qual medida causou o alerta.
- **Mahalanobis:** considera todas as medidas juntas.
- **Isolation Forest:** isola pontos raros usando árvores.
- **Regra fixa atual do projeto:** inclinação acima de 18 graus ou simetria abaixo de 80%. Como o código não define a simetria, ela foi tomada aqui como 100% menos a assimetria do passo.

In [ ]:
class RobustZ:
    def fit(self, values):
        self.median = np.median(values, axis=0)
        q25, q75 = np.percentile(values, [25, 75], axis=0)
        self.scale = np.maximum((q75 - q25) / 1.349, 1e-6)
        return self

    def score(self, values):
        return np.abs((values - self.median) / self.scale).max(axis=1)


class Mahalanobis:
    def fit(self, values):
        self.mean = values.mean(axis=0)
        self.precision = LedoitWolf().fit(values).precision_
        return self

    def score(self, values):
        centered = values - self.mean
        return np.sqrt(np.einsum('ij,jk,ik->i', centered, self.precision, centered))


class IForest:
    def fit(self, values):
        self.model = IsolationForest(n_estimators=200, random_state=SEED).fit(values)
        return self

    def score(self, values):
        return -self.model.score_samples(values)


class CurrentRule:
    fixed_threshold = 1.0  # passa de 1 quando a regra atual dispararia

    def fit(self, values):
        return self

    def score(self, values):
        tilt = values[:, NAMES.index('tronco_frente_media')] / 18
        asymmetry = values[:, NAMES.index('assimetria_passo')] / 0.20
        return np.maximum(tilt, asymmetry)


METHODS = {'z_robusto': RobustZ, 'mahalanobis': Mahalanobis, 'isolation_forest': IForest, 'regra_atual': CurrentRule}


def walk_scores(detector, df):
    scores = pd.Series(detector.score(df[NAMES].to_numpy()), index=df['trial'].to_numpy())
    return scores.groupby(level=0).median()


def threshold_from_new_people(method, train):
    if hasattr(method, 'fixed_threshold'):
        return method.fixed_threshold
    pooled = []
    for person in sorted(train['pid'].unique()):
        detector = method().fit(train.loc[train['pid'] != person, NAMES].to_numpy())
        pooled.append(walk_scores(detector, train[train['pid'] == person]))
    return float(np.quantile(pd.concat(pooled), 1 - FALSE_ALARM_TARGET))


def validate(method):
    false_alarm, false_alarm_turns = [], []
    detection = {name: [] for name in SIMULATIONS}
    auroc = {name: [] for name in SIMULATIONS}
    for person in participants:
        train = straight_df[straight_df['pid'] != person]
        normal_rows = straight_df[straight_df['pid'] == person]
        if normal_rows.empty:
            continue
        threshold = threshold_from_new_people(method, train)
        detector = method().fit(train[NAMES].to_numpy())
        normal = walk_scores(detector, normal_rows)
        false_alarm.append(float((normal > threshold).mean()))
        turns = windows_df[(windows_df['pid'] == person) & (windows_df['sequence'] != 'straight')]
        if not turns.empty:
            false_alarm_turns.append(float((walk_scores(detector, turns) > threshold).mean()))
        for name, df in simulated.items():
            changed = walk_scores(detector, df[df['pid'] == person])
            detection[name].append(float((changed > threshold).mean()))
            labels = np.r_[np.zeros(len(normal)), np.ones(len(changed))]
            auroc[name].append(float(roc_auc_score(labels, np.r_[normal.to_numpy(), changed.to_numpy()])))
    return {
        'falso_alarme_linha_reta': float(np.mean(false_alarm)),
        'falso_alarme_pior_pessoa': float(np.max(false_alarm)),
        'falso_alarme_curvas': float(np.mean(false_alarm_turns)),
        'deteccao': {name: float(np.mean(values)) for name, values in detection.items()},
        'AUROC': {name: float(np.mean(values)) for name, values in auroc.items()},
    }


comparison = {}
for method_name, method in METHODS.items():
    print('Avaliando', method_name, '...')
    comparison[method_name] = validate(method)

print('\nAlarmes falsos em pessoas saudáveis que ficaram fora do treino (meta: 5%):')
print(pd.DataFrame({
    name: {
        'linha reta': result['falso_alarme_linha_reta'],
        'pior pessoa': result['falso_alarme_pior_pessoa'],
        'curvas': result['falso_alarme_curvas'],
    }
    for name, result in comparison.items()
}).round(3).to_string())
print('\nDetecção das anomalias simuladas:')
print(pd.DataFrame({name: result['deteccao'] for name, result in comparison.items()}).round(3).to_string())

## 7. Referência final

O método escolhido é a **pontuação z robusta**. Na comparação, ele foi o que melhor equilibrou detecção e alarmes falsos, e o Isolation Forest deixou passar desvios fortes em uma única medida. Ele também explica o alerta, o que ajuda no laudo.

A escolha usou as mesmas simulações do teste, o que reforça que esses números são otimistas.

A referência é salva em JSON, então o backend pode usá-la sem depender da versão das bibliotecas do Colab.

In [ ]:
final_detector = RobustZ().fit(straight_df[NAMES].to_numpy())
final_threshold = threshold_from_new_people(RobustZ, straight_df)

reference = {
    'metodo': 'z_robusto',
    'medidas': NAMES,
    'descricao': MEASURES,
    'mediana': dict(zip(NAMES, final_detector.median.round(6).tolist())),
    'escala': dict(zip(NAMES, final_detector.scale.round(6).tolist())),
    'limiar': round(final_threshold, 6),
    'taxa_amostragem_hz': FS,
    'janela_s': WINDOW_SECONDS,
    'passo_janela_s': HOP_SECONDS,
    'fonte': 'Multi-Gait-Posture 1.0.0, caminhadas em linha reta com andador, 14 adultos saudáveis.',
}
(OUTPUT_DIR / 'marcha_referencia.json').write_text(json.dumps(reference, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Limiar:', reference['limiar'])
print(pd.DataFrame({'mediana': reference['mediana'], 'escala': reference['escala']}).round(3).to_string())


def assess_walk(X, reference):
    rows = walk_windows(X)
    if not rows:
        return {'avaliada': False, 'motivo': 'caminhada curta demais ou sem passos suficientes'}
    measures = pd.DataFrame(rows)[reference['medidas']]
    median = np.array([reference['mediana'][name] for name in reference['medidas']])
    scale = np.array([reference['escala'][name] for name in reference['medidas']])
    distances = np.abs((measures.to_numpy() - median) / scale)
    score = float(np.median(distances.max(axis=1)))
    per_measure = dict(zip(reference['medidas'], np.median(distances, axis=0)))
    outside = [name for name, value in sorted(per_measure.items(), key=lambda kv: -kv[1]) if value > reference['limiar']]
    return {'avaliada': True, 'pontuacao': round(score, 2), 'alerta': score > reference['limiar'], 'medidas_fora_da_faixa': outside}


# Só uma demonstração do formato: estas caminhadas fazem parte da referência final.
example = next(t for t in straight_trials if t['speed'] == 0.5)
print('\nCaminhada original:', assess_walk(example['X'], reference))
print('Mesma caminhada com o tronco 20 graus à frente:', assess_walk(rotate_upper_body(example['X'], forward=20), reference))

## 8. Salvar as métricas

In [ ]:
import scipy
import sklearn

metrics = {
    'base': 'A multi-camera and multimodal dataset for posture and gait analysis 1.0.0',
    'amostra': {
        'participantes': len(participants),
        'caminhadas': len(trials),
        'janelas': windows_df.groupby('sequence').size().to_dict(),
    },
    'validacao': 'Uma pessoa fora por vez; limiar definido sem olhar a pessoa testada; pontuação da caminhada é a mediana das janelas.',
    'meta_falso_alarme': FALSE_ALARM_TARGET,
    'metodo_escolhido': 'z_robusto',
    'comparacao': comparison,
    'medianas_saudaveis_por_velocidade': straight_df.groupby('speed')[NAMES].median().round(3).to_dict(),
    'limitacoes': [
        'Só adultos jovens e saudáveis, usando andador, em velocidades lentas (0,3, 0,5 e 0,7 m/s); não há casos reais de marcha alterada.',
        'As anomalias são simuladas e alteram o que as próprias medidas observam; a detecção é otimista e não prova desempenho em pacientes.',
        'O método foi escolhido usando as mesmas simulações do teste.',
        'A referência vem de sensores inerciais; o uso com MediaPipe em vídeo ainda não foi validado.',
        'A base não calcula nem valida métricas de marcha; as medidas foram calculadas neste notebook.',
        'Algumas pessoas saudáveis geram mais alarmes falsos; veja falso_alarme_pior_pessoa.',
        'Apoio à decisão, não diagnóstico.',
    ],
    'versoes': {'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__, 'scikit-learn': sklearn.__version__},
}
(OUTPUT_DIR / 'marcha_metricas.json').write_text(json.dumps(metrics, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')

print('Arquivos em', OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {path.name} ({path.stat().st_size / 1024:.1f} KB)')

## 9. Próximos passos

Baixe de `Meu Drive/omnicare/resultados_marcha/`:

- `marcha_referencia.json`: coloque em `backend/models/`.
- `marcha_metricas.json`: use no relatório técnico, junto com as limitações listadas nele.